In [1]:
import pandas as pd

In [2]:
race_data = pd.read_csv('../data/lap_weather_data_2018_2025.csv')

In [3]:
# Rank races by the share of laps marked as rainy
rainy_races = race_data.groupby(['Year', 'EventName'])['Rainfall'].transform('any')

race_rain_pct = (
    race_data[rainy_races]
    .groupby(['Year', 'EventName'])['Rainfall']
    .mean()
    .mul(100)
    .round(2)
    .loc[lambda s: s > 10]
    .reset_index(name='PercentRain')
    .sort_values('PercentRain', ascending=False)
    .reset_index(drop=True)
)

race_rain_pct

,Year,EventName,PercentRain
0,2019,German Grand Prix,98.77
1,2022,Japanese Grand Prix,90.34
2,2019,Monaco Grand Prix,83.56
3,2018,Spanish Grand Prix,82.37
4,2024,São Paulo Grand Prix,72.31
5,2021,Belgian Grand Prix,66.67
6,2024,British Grand Prix,42.29
7,2022,Monaco Grand Prix,38.25
8,2021,Emilia Romagna Grand Prix,32.12
9,2025,Australian Grand Prix,29.56


In [4]:
race_data['LapTime'] = pd.to_timedelta(race_data['LapTime']).dt.total_seconds()

In [5]:
# Compare each 2024 British GP lap to that lap's field median
median_laps = (
    race_data.loc[
        (race_data['Year'] == 2024)
        & (race_data['EventName'] == 'British Grand Prix')
        & (race_data['TrackStatus'] == 1)
        & (race_data['IsPitLap'] == False)
    ]
    .groupby('LapNumber', as_index=False)['LapTime']
    .median()
    .rename(columns={'LapTime': 'MedianLapTime'})
)

driver_laps = (
    race_data.loc[
        (race_data['Year'] == 2024)
        & (race_data['EventName'] == 'British Grand Prix')
        & (race_data['TrackStatus'] == 1)
        & (race_data['IsPitLap'] == False)
        & (race_data['LapNumber'].isin(median_laps['LapNumber']))
    ][['Driver', 'LapNumber', 'LapTime', 'Rainfall']]
    .reset_index(drop=True)
)

driver_laps_diff = driver_laps.merge(
    median_laps,
    on='LapNumber',
    how='left'
)

driver_laps_diff['LapTimeDiff'] = (
    driver_laps_diff['LapTime'] 
    - driver_laps_diff['MedianLapTime']
)

In [7]:
# Measure which drivers improved most relative to the field in rain
rain_performance = (
    driver_laps_diff.loc[driver_laps_diff['Rainfall'] == True]
    .groupby('Driver')['LapTimeDiff']
    .mean()
    .reset_index(name='AvgRainDiff')
)

dry_performance = (
    driver_laps_diff.loc[driver_laps_diff['Rainfall'] == False]
    .groupby('Driver')['LapTimeDiff']
    .mean()
    .reset_index(name='AvgDryDiff')
)

driver_performance = rain_performance.merge(
    dry_performance,
    on='Driver'
)

driver_performance['WeatherGain'] = (
    driver_performance['AvgDryDiff']
    - driver_performance['AvgRainDiff']
)

driver_performance.sort_values('WeatherGain', ascending=False).reset_index(drop=True)

,Driver,AvgRainDiff,AvgDryDiff,WeatherGain
0,NOR,-1.495342,-0.745586,0.749756
1,PIA,-1.432500,-0.995467,0.437033
2,HAM,-1.327250,-1.004700,0.322550
3,BOT,1.308353,1.479250,0.170897
4,VER,-0.979361,-0.921267,0.058094
5,TSU,0.322237,0.376759,0.054522
6,MAG,0.428278,0.469733,0.041456
7,HUL,-0.024868,0.000448,0.025317
8,OCO,1.636800,1.648185,0.011385
9,SAI,-0.638289,-0.653907,-0.015618
